In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

data = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")

X_train_origin = data.drop(['SalePrice'], axis=1)
y = data.SalePrice
X_test_origin = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
X_train = X_train_origin.copy()
X_test = X_test_origin.copy()

#### Drop column with >10% of missing data

In [2]:
threshold = 0.1
total = X_train.isnull().sum().sort_values(ascending = False)
percent = (X_train.isnull().sum()/X_train.isnull().count()).sort_values(ascending = False)
missing_data = pd.concat([total, percent], axis =1, keys=['Total', 'Percentage'])
display(missing_data[(missing_data.Percentage > 0)])
X_train = X_train.drop(missing_data[missing_data.Percentage > threshold].index, axis = 1)
X_test = X_test.drop(missing_data[missing_data.Percentage > threshold].index, axis = 1)

,Total,Percentage
PoolQC,1453,0.995205
MiscFeature,1406,0.963014
Alley,1369,0.937671
Fence,1179,0.807534
FireplaceQu,690,0.472603
LotFrontage,259,0.177397
GarageCond,81,0.055479
GarageType,81,0.055479
GarageYrBlt,81,0.055479
GarageFinish,81,0.055479


#### Drop column with >95% of unique

In [3]:
threshold = 0.95
total = X_train.nunique()
percent = (X_train.nunique()/X_train.count())#.sort_values(ascending = False)
idness_data = pd.concat([total, percent], axis =1, keys=['Total', 'Percentage'])
display(idness_data[(idness_data.Percentage > 0.2)].sort_values(by='Percentage', ascending = False))
X_train = X_train.drop(idness_data[idness_data.Percentage > threshold].index, axis = 1)
X_test = X_test.drop(idness_data[idness_data.Percentage > threshold].index, axis = 1)


,Total,Percentage
Id,1460,1.000000
LotArea,1073,0.734932
GrLivArea,861,0.589726
BsmtUnfSF,780,0.534247
1stFlrSF,753,0.515753
TotalBsmtSF,721,0.493836
BsmtFinSF1,637,0.436301
GarageArea,441,0.302055
2ndFlrSF,417,0.285616
MasVnrArea,327,0.225207


#### Handle remaining missing data

In [4]:
# Combine Test and Training sets to maintain consistancy.
combined = pd.concat([X_train,X_test],axis=0)
combined.head()
combined.shape

# Missing Value Handling
def HandleMissingValues(df):
    # for Object columns fill using 'UNKOWN'
    # for Numeric columns fill using median
    num_cols = [cname for cname in df.columns if df[cname].dtype in ['int64', 'float64']]
    cat_cols = [cname for cname in df.columns if df[cname].dtype == "object"]
    values = {}
    for a in cat_cols:
        values[a] = 'UNKOWN'

    for a in num_cols:
        values[a] = df[a].median()
        
    df.fillna(value=values,inplace=True)
    
HandleMissingValues(combined)
combined.head()

# Check for any missing values
combined.isnull().sum().sum()

0

In [5]:
#Categorical Feature Encoding
def getObjectColumnsList(df):
    return [cname for cname in df.columns if df[cname].dtype == "object"]

def PerformOneHotEncoding(df,columnsToEncode):
    return pd.get_dummies(df,columns = columnsToEncode)

cat_cols = getObjectColumnsList(combined)
combined = PerformOneHotEncoding(combined,cat_cols)
combined.head()

,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,...,SaleType_New,SaleType_Oth,SaleType_UNKOWN,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,60,8450,7,5,2003,2003,196.0,706.0,0.0,150.0,...,0,0,0,1,0,0,0,0,1,0
1,20,9600,6,8,1976,1976,0.0,978.0,0.0,284.0,...,0,0,0,1,0,0,0,0,1,0
2,60,11250,7,5,2001,2002,162.0,486.0,0.0,434.0,...,0,0,0,1,0,0,0,0,1,0
3,70,9550,7,5,1915,1970,0.0,216.0,0.0,540.0,...,0,0,0,1,1,0,0,0,0,0
4,60,14260,8,5,2000,2000,350.0,655.0,0.0,490.0,...,0,0,0,1,0,0,0,0,1,0


In [6]:
#respliting the data into train and test datasets
X_train=combined.iloc[:1460,:]
X_test=combined.iloc[1460:,:]
print(X_train.shape)
print(X_test.shape)

(1460, 287)
(1459, 287)


In [7]:
import xgboost as xgb

model_xgb = xgb.XGBRegressor(n_estimators=340, max_depth=2, learning_rate=0.2)
model_xgb.fit(X_train, y)
xgb_preds=model_xgb.predict(X_test)

/opt/conda/lib/python3.6/site-packages/xgboost/core.py:587: FutureWarning: Series.base is deprecated and will be removed in a future version
  if getattr(data, 'base', None) is not None and \
/opt/conda/lib/python3.6/site-packages/xgboost/core.py:588: FutureWarning: Series.base is deprecated and will be removed in a future version
  data.base is not None and isinstance(data, np.ndarray) \


[10:14:41] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.


In [8]:
#make the submission data frame
submission = {
    'Id': X_test_origin.Id.values,
    'SalePrice': xgb_preds
}
solution = pd.DataFrame(submission)
solution.head()

,Id,SalePrice
0,1461,119926.500000
1,1462,169705.906250
2,1463,184655.890625
3,1464,188075.765625
4,1465,200031.843750


In [9]:
#make the submission file
solution.to_csv('submission.csv',index=False)